In [1]:
# Import packages 
import pandas as pd
import sys
import matplotlib.pyplot as plt
from collections import defaultdict, OrderedDict
import glob
import os
import shutil
import numpy as np
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import glob
import sys
import re
print(sys.version)
sys.path.append('/Users/geba9152/LIET/liet/')

from liet_res_class import FitParse, fitparse_intersect
import rnap_lib_data_sim as ds
import rnap_lib_plotting as rp
from rnap_lib_fitting_results import results_loader, pdf_generator, hist_generator

import numpy as np
import scipy as sp
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec


# LIET library packages
sys.path.append('/Users/geba9152/LIET/liet/')
import rnap_lib_data_proc as dp
from liet_res_class import FitParse, fitparse_intersect
from rnap_lib_fitting_results import results_loader
import rnap_lib_plotting as rp

sys.path.append('/Users/geba9152/fall_2024_3prime/')
from load_in_liet import pull_out_params_meta_samples

3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:36:13) [GCC 12.3.0]


# Parsing MANE Annotation Files

In [ ]:
# 1. Make 5p truncated file for coverage calculations
# According to README in NCBI folder, check to be sure on this:
# Ru takes 1/3 of the gene length and subtracts it from the 5p end

transcript_bed_dir = '/scratch/Shares/dowell/geba9152/MANE/transcript.MANE.refseq.bed'
outdir = '/scratch/Shares/dowell/geba9152/MANE/'

def make_5p_trunc_file(transcript_bed_dir, outdir):
    
    # Read in MANE transcript file
    MANE_transcript = pd.read_csv(transcript_bed_dir, sep = "\t", header = None, names = ['chr','start','stop','ROI','.','strand'])

    # Calculate MANE length 
    MANE_transcript['length'] = abs(MANE_transcript['start'] - MANE_transcript['stop'])
    
    # Calculate 1/3 MANE length 
    MANE_transcript['one_third'] = (MANE_transcript['length'] / 3).astype(int)

    # Calculate 5p trunc length, rule won't pertain to tiny genes (>300 bps) according to Ru
    # Max of 750
    MANE_transcript['trunc_distance_5p'] = MANE_transcript.apply(
        lambda row: min(row['one_third'], 750) if row['length'] > 300 else 0,
        axis=1
    )
    
    ### Calculate start and stops ###
    MANE_transcript['start-5p-trunc'] = MANE_transcript.apply(
        lambda row: row['start'] + row['trunc_distance_5p'] if row['strand'] == '+' else row['start'],
        axis=1
    )
    
    MANE_transcript['stop-5p-trunc'] = MANE_transcript.apply(
        lambda row: row['stop'] - row['trunc_distance_5p'] if row['strand'] == '-' else row['stop'],
        axis=1
    )
    
    # Organize #
    MANE_transcript = MANE_transcript[['chr','start-5p-trunc','stop-5p-trunc','ROI','.','strand']]
    MANE_transcript = MANE_transcript.sort_values(by=['chr','start-5p-trunc'])
    
    # Save
    MANE_transcript.to_csv(outdir + "5p_trunc_transcripts.MANE.refseq.bed", sep = "\t", header = None, index = None)
    
    return MANE_transcript
    
make_5p_trunc_file(transcript_bed_dir, outdir)


Coverage calculated in all samples using bedtools coverage. Bedtools coverage ran on all MANE transcripts downloaded from here: https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/
Version: MANE.GRCh38.v1.4.refseq_genomic.gtf.gz (pulling for only transcript feature)

Coverage script: https://github.com/GEFB2000/LIET-3end-analysis/blob/main/gene_curation_MANE/scripts/calculate-coverage.sbatch

Coverage data: `~/3end_paper_backed_up_data/bedtools-coverage/`

# Annotation isolation filter

Annotation isolation was performed using bedtools slop (to add on padding regions to bedfile) & bedtools merge (to remove overlapping trancripts), see `annotation-isolation-filter.sbatch`

# Dealing with manually curated sets
* Added in genes that were manually curated from Stanley et al. 2025 NAR & a few extra manually curated genes
* Genes were added after annotation isolation filter & before all other filters to ensure they were optimal for LIET

In [25]:
# Dealing with manually curated sets #
# Jacob -- list #1; from Stanley et al. 2025 NAR
# Andrew -- extra manually curated genes

##############################
## 3p/5p UTR file GT GENES (Jacob) ## 
############################### 
# Based on transcripts bed in ncbi folder on scratch

# Getting rid of bad genes in 5p/3p UTR files
gt_list = pd.read_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/chr1-6-5p-UTR.liet.ann", sep = "\t", header = None)
gt_list.columns = ['chr', 'start', 'stop', 'ROI', 'length', 'strand']

print(len(gt_list))
# Get rid of bad genes #
gt_list = gt_list[~gt_list['ROI'].isin(genes_to_remove)]
# gt_list.to_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/chr1-6-5p-UTR.liet.ann", sep = "\t", header = None, index = None)
gt_list

# Getting rid of bad genes in 5p UTR pad file
gtpad = pd.read_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/pads/chr1-6-5p-UTR.pad", sep = "\t", header = None)
gtpad.columns = ['ROI', 'pad']
gtpad = gtpad[~gtpad['ROI'].isin(genes_to_remove)]
print(len(gtpad))
gtpad.to_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/pads/chr1-6-5p-UTR.pad", sep = "\t", header = None, index = None)


###################################
## 3p UTR file GT GENES (Andrew HCT) ## 
###################################
# Based on transcripts bed in ncbi folder on scratch

# Saving 5p/3p UTR files for Andrew's set
# 3p setUTR
andrewgenes = pd.read_csv("/Users/geba9152/fall_2024_3prime/andrew-new-genes.csv", sep = ",", header = None)
isoforms = pd.read_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/0.1-cov-filtered-MANE-plus_ground_truth-3p-UTR.liet.ann", sep = "\t", header = None)
isoforms[7] = isoforms[3].str.split("|").str[1].str.split(".").str[0]
isoforms[6] = isoforms[3].str.split("|").str[0]
isoforms.columns = ['chr', 'start', 'stop', 'full', 'length', 'strand', 'ROI', 'gene']
isoforms = isoforms[['chr', 'start', 'stop', 'strand', 'full','ROI', 'gene']]
andrewgenes.columns = ['gene']
andrewgenes = andrewgenes.merge(isoforms, on = 'gene')
andrewgenes = andrewgenes[['chr', 'start', 'stop', 'full', 'strand']]
andrewgenes['length'] = abs(andrewgenes['start'] - andrewgenes['stop'])
andrewgenes = andrewgenes[['chr', 'start', 'stop', 'full', 'length', 'strand']]
andrewgenes = andrewgenes.drop_duplicates(subset="full")
andrewgenes = andrewgenes.sort_values(by = ['chr','start'])
andrewgenes.to_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/Andrew-HCT-Manually-Inspected-3p-UTR.liet.ann", sep = "\t", header = None, index = None)
filtered_genes = andrewgenes['full'].to_list()
andrewgenes.columns = ['chr', 'start', 'stop', 'ROI', 'length', 'strand']

# 5p UTR set
outdir = '/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/'

def organize_andrew_set(andrewgenes, outdir):
    ann_ex = pd.read_csv("/scratch/Shares/dowell/genomes/hg38/ncbi/hg38_refseq_exons.bed", sep = "\t", header = None, names = ['chr','start','stop','ROI','.','strand'])

    last_exon_coordinates = pd.DataFrame(columns=ann_ex.columns)

    # select only isoforms we are interested in
    ann_ex["ROI"] = ann_ex["ROI"].str.replace(":", "|")
    selected_ann_ex = ann_ex.loc[ann_ex["ROI"].isin(filtered_genes)]

    # loop through df (group) and gene (roi)
    for roi, group in selected_ann_ex.groupby('ROI'):
        if group['strand'].iloc[0] == "+":
            sorted_stops = group.sort_values(by='stop', ascending=False)
            # Note "last_exon == UTR"
            last_exon = sorted_stops.iloc[0]

        elif group['strand'].iloc[0] == "-":
            sorted_starts = group.sort_values(by='start', ascending=True)
            last_exon = sorted_starts.iloc[0]

        # Concatenate the selected exon to the DataFrame
        last_exon_coordinates = pd.concat([last_exon_coordinates, pd.DataFrame([last_exon])], ignore_index=True)

    # Organize after all the concatenations
    last_exon_coordinates = last_exon_coordinates.sort_values(by=["chr", "start"])
    last_exon_coordinates.columns = ["chr","start-ex","stop-ex","ROI",".","strand"]
    last_exon_coordinates = last_exon_coordinates[["start-ex","stop-ex","ROI"]]
    
    # Merge with LIET annotation to get correct structure
    fivep_UTR = last_exon_coordinates.merge(andrewgenes, on="ROI")
    fivep_UTR = fivep_UTR.drop_duplicates(subset="ROI")
    
    # Make copy for pad 
    fivep_UTR_pad = fivep_UTR.copy()

    fivep_UTR['start'] = fivep_UTR.apply(lambda row: row["stop-ex"] if row["strand"] == "-" else row['start'], axis=1)
    fivep_UTR['stop'] = fivep_UTR.apply(lambda row: row["start-ex"] if row["strand"] == "+" else row['stop'], axis=1)

    # Check that len of 5p and 3p files are same
    if len(andrewgenes) != len(fivep_UTR):
        raise ValueError(
        f"Length mismatch: threep_UTR has {len(andrewgenes)} rows, "
        f"but fivep_UTR result has {len(fivep_UTR)} rows."
    )

    fivep_UTR_ann = fivep_UTR[['chr','start','stop','ROI','length','strand']]
    fivep_UTR_ann = fivep_UTR_ann.sort_values(by=['chr', 'start'])

    # save
    output_file = os.path.join(outdir, f"Andrew-HCT-Manually-Inspected-5p-UTR.liet.ann")
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    fivep_UTR_ann.to_csv(output_file, index=False, sep="\t", header=None)

    # redorder & sort for bed format
    fivep_UTR_ann['.'] = '.'
    filtered_genes_bed = fivep_UTR_ann[['chr','start','stop','ROI','.','strand']]
    filtered_genes_bed = filtered_genes_bed.sort_values(by=['chr', 'start'])

    # save
    output_file = os.path.join(outdir, f"Andrew-HCT-Manually-Inspected-5p-UTR.bed")
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    filtered_genes_bed.to_csv(output_file, index=False, sep="\t", header=None)

    # pad files 
    fivep_UTR_pad['UTR-len'] = fivep_UTR_pad.apply(
        lambda row: abs(row["stop"] - row["start-ex"]) if row["strand"] == "+" else abs(row["stop-ex"] - row["start"]),
        axis=1
    )

    # Calculate pads
    fivep_UTR_pad['3prime-end-pad'] = fivep_UTR_pad['UTR-len'] + 20000
    fivep_UTR_pad['5prime-end-pad'] = 3000
    fivep_UTR_pad['pad'] = fivep_UTR_pad['5prime-end-pad'].astype(str) + ',' + fivep_UTR_pad['3prime-end-pad'].astype(str)

    # Sort and save 
    # Reorder fivep_UTR_pad to match the gene order in fivep_UTR_ann
    fivep_UTR_pad = fivep_UTR_pad.set_index('ROI')
    fivep_UTR_pad = fivep_UTR_pad.loc[fivep_UTR_ann['ROI']].reset_index()
    fivep_UTR_pad = fivep_UTR_pad[['ROI','pad']]
    
    # Check that len of pad and 5p files are same
    if len(fivep_UTR_pad) != len(fivep_UTR):
        raise ValueError(
        f"Length mismatch: fivep_UTR_pad has {len(fivep_UTR_pad)} rows, "
        f"but fivep_UTR result has {len(fivep_UTR)} rows."
    )

    output_file = os.path.join(outdir, f"Andrew-HCT-Manually-Inspected-5p-UTR.pad")
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    fivep_UTR_pad.to_csv(output_file, index=False, sep="\t", header=None)

organize_andrew_set(andrewgenes, outdir)


In [ ]:
## CREATE AN ANNOTATION FILE WITH UNIQUE ANNOTATIONS ABOVE + MANE ##
MANE_transcript_annotation_file = '/scratch/Shares/dowell/geba9152/MANE/transcript.MANE.refseq.bed'
MANE_exon_annotation_file = '/scratch/Shares/dowell/geba9152/MANE/exon.MANE.refseq.bed'

RefSeq_transcript_annotation_file = '/scratch/Shares/dowell/genomes/hg38/ncbi/hg38_refseq_transcripts.bed'
RefSeq_exon_annotation_file = '/scratch/Shares/dowell/genomes/hg38/ncbi/hg38_refseq_exons.bed'

andrew_genes = '/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/Andrew-HCT-Manually-Inspected-3p-UTR.liet.ann'
jacob_genes = '/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/chr1-6-3p-UTR.liet.ann'

def create_master_exon_transcript_anns():

    # Load MANE annotations
    transcript_ann = pd.read_csv(
        MANE_transcript_annotation_file, 
        sep="\t", 
        header=None, 
        names=['chr', 'start', 'stop', 'gene', '.', 'strand']
    )
    exon_ann = pd.read_csv(
        MANE_exon_annotation_file, 
        sep="\t", 
        header=None, 
        names=['chr', 'start', 'stop', 'gene', '.', 'strand']
    )

    # Load RefSeq annotations
    refseq_transcripts = pd.read_csv(
        RefSeq_transcript_annotation_file, 
        sep="\t", 
        header=None, 
        names=['chr', 'start', 'stop', 'gene', '.', 'strand']
    )
    refseq_exons = pd.read_csv(
        RefSeq_exon_annotation_file, 
        sep="\t", 
        header=None, 
        names=['chr', 'start', 'stop', 'gene', '.', 'strand']
    )

    # Load Andrew and Jacob annotations
    andrew_ann = pd.read_csv(
        andrew_genes, 
        sep="\t", 
        header=None, 
        names=['chr', 'start', 'stop', 'gene', '.', 'strand']
    )
    jacob_ann = pd.read_csv(
        jacob_genes, 
        sep="\t", 
        header=None, 
        names=['chr', 'start', 'stop', 'gene', '.', 'strand']
    )
    
    transcript_ann_duplicates = transcript_ann[transcript_ann['gene'].duplicated(keep=False)]
    jacob_ann_duplicates = jacob_ann[jacob_ann['gene'].duplicated(keep=False)]
    andrew_ann_duplicates = andrew_ann[andrew_ann['gene'].duplicated(keep=False)]
    
    # Combine Andrew and Jacob annotations
    extra_annotations = pd.concat([andrew_ann, jacob_ann]).drop_duplicates()
    extra_annotations_duplicates = extra_annotations[extra_annotations['gene'].duplicated(keep=False)]
    print(f"extra ann dups{len(extra_annotations_duplicates)} ")

    extra_annotations['.'] = '.'
    print(f"Original # transcripts {len(transcript_ann)}")
    
    # Add a source column to identify the origin
    extra_annotations['source'] = 'extra'
    transcript_ann['source'] = 'transcript'

    # Combine the dataframes
    master_transcript = pd.concat([extra_annotations, transcript_ann])
    
    # Pull out duplicates
    dups = master_transcript[master_transcript['gene'].duplicated(keep=False)]
    dups_genes = dups['gene'].to_list()
        
    # Drop dups from master transcript
    master_transcript = master_transcript[~master_transcript['gene'].isin(dups_genes)]
    
    # Sort dups/pull out mane transcripts
    dups = dups.sort_values(by=['gene', 'source'])
    keepers = dups[dups['source'] == 'transcript']
    
    # Concatenate back into master transcript
    master_transcript = pd.concat([master_transcript, keepers])
    dups = master_transcript[master_transcript['gene'].duplicated(keep=False)]   
    print(len(dups))
    
    # Sort and obtain correct columns
    master_transcript = master_transcript.sort_values(by=['chr', 'start'])
    master_transcript = master_transcript[['chr','start','stop','gene','.','strand']]

    print(f"Total # transcripts {len(master_transcript)}")
    
    # master exons #
    # get genes we care about from refseq exon file 
    extra_annotations['gene'] = extra_annotations['gene'].str.replace("|",":")
    extra_annotations_list = extra_annotations['gene'].to_list()
    refseq_exons = refseq_exons[refseq_exons['gene'].isin(extra_annotations_list)]
    refseq_exons['gene'] = refseq_exons['gene'].str.replace(":","|")
    
    # Identify the unique genes in exon_ann
    genes_in_exon_ann = set(exon_ann['gene'])

    # Separate exons from refseq_exons for genes NOT in exon_ann
    refseq_exons_to_keep = refseq_exons[~refseq_exons['gene'].isin(genes_in_exon_ann)]

    # Combine exons: all from exon_ann + remaining from refseq_exons
    master_exons = pd.concat([exon_ann, refseq_exons_to_keep])

    # Ensure the final DataFrame is sorted by chromosome and start position
    master_exons = master_exons.sort_values(by=['chr', 'start'])
        
    # Save master files
    # Andrew/Jacob are ncbi RefSeq based (most upstream TCS for isoform selection)
    # Others are MANE transcripts
    master_transcript_file = "/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/annotations/master_MANE_Andrew_Jacob_transcripts.bed"
    master_exon_file = "/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/annotations/master_MANE_Andrew_Jacob_exons.bed"

    master_transcript.to_csv(master_transcript_file, sep="\t", index=False, header=False)
    master_exons.to_csv(master_exon_file, sep="\t", index=False, header=False)

# Call the function
create_master_exon_transcript_anns()

# Coverage Filter

In [49]:
# 1. Perform coverage + complexity filtering on gene list

# Load in isolated genes
isolated_genes = pd.read_csv("/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/annotations/isolated-3000-5p-20000-3p-transcript.MANE.sorted.bed", sep = "\t", header = None)
isolated_genes = isolated_genes[3].to_list()

# Set outdir
output_dir = "/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/filtered-genes/"
cov_directory = "/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/bedtools-coverage/"

def coverage_filter_genes(metadata_df, isolated_genes, cov_directory, output_dir):

    for _, experiment in metadata_df.iterrows():
        experiment_name = experiment['Experiment']
        samples = experiment['Names'].split(',')
        coverage_threshold = experiment['Sense_Coverage_Cutoff']
        complexity_cutoff = experiment['Normalized_Complexity_Cutoff']
        
        print(f"Processing experiment: {experiment_name}")
        
        # Initialize variable
        gene_sets = []  # To store genes passing coverage for each sample

        # Process each sample
        for sample in samples:
            coverage_file = os.path.join(cov_directory, experiment_name, f"{sample}.3p.combined.txt")
            if not os.path.exists(coverage_file):
                print(f"Coverage file not found for sample {sample}, skipping.")
                continue

            # Read coverage file
            df = pd.read_csv(coverage_file, sep="\t", header=None)
            
            # Jacob's list
            jacob = pd.read_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/chr1-6-3p-UTR.liet.ann", sep = "\t", header = None)
            jacobList = jacob[3].to_list()
            
#             print(len(jacobList))
#             print(len(isolated_genes))
#             print(len(isolated_genes) + len(jacobList))
            
            # Andrew's list (HCT specific)
            andrew = pd.read_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-annotations/lietanns/Andrew-HCT-Manually-Inspected-3p-UTR.liet.ann", sep = "\t", header = None)
            andrewList = andrew[3].to_list()
            
            # Filter genes by coverage threshold
            df_threshold = df[df[9] >= coverage_threshold]
            
            # Add in complexity filter here
            df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
            df_threshold = df_threshold[df_threshold['norm_complexity'] >= complexity_cutoff]
                        
            ## ADDING IN OLD GENES HERE ##
            # Adding in jacob/andrew
            isolated_plus_jacob_andrew_genes = isolated_genes + jacobList + andrewList

            # Retain only filtered genes
            filtered_genes = set(df_threshold[3]).intersection(isolated_plus_jacob_andrew_genes)
            print(f"Sample {sample} - Genes after coverage filtering: {len(filtered_genes)}")

            gene_sets.append(filtered_genes)

        # Find common genes across all samples in the experiment
        common_genes = set.intersection(*gene_sets) if gene_sets else set()
        print(f"Sample {experiment_name} - TOTAL genes after ALL coverage filtering: {len(common_genes)}")

        # Save the final list of genes for the experiment
        output_file = os.path.join(output_dir, f"{experiment_name}/isolated_{coverage_threshold}_coverage_filtered_genes.txt")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        pd.Series(list(common_genes)).to_csv(output_file, index=False, sep="\t", header=None)

coverage_filter_genes(metadata_df, isolated_genes, cov_directory, output_dir)

Processing experiment: Meta-Eric-IFN-Beta
Sample meta_PRO-BSA-Eric - Genes after coverage filtering: 2625
Sample meta_PRO-IFNB-Eric - Genes after coverage filtering: 2692
Sample Meta-Eric-IFN-Beta - TOTAL genes after ALL coverage filtering: 2582


/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

Processing experiment: OV90-SRRs
Sample DMSO_0_R1 - Genes after coverage filtering: 2405
Sample DMSO_0_R2 - Genes after coverage filtering: 2680
Sample OV90-SRRs - TOTAL genes after ALL coverage filtering: 2387
Processing experiment: Daniel-IFN


/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

Sample PRO-BSA-Human-1 - Genes after coverage filtering: 2507
Sample PRO-BSA-Human-2 - Genes after coverage filtering: 2551
Sample PRO-IFN-Human-1 - Genes after coverage filtering: 2448


...

Sample 1_MOLM14_V - Genes after coverage filtering: 2669
Sample 4_MOLM14_V - Genes after coverage filtering: 2692
Sample 2_MOLM14_1H - Genes after coverage filtering: 2715


/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

Sample 5_MOLM14_1H - Genes after coverage filtering: 2700
Sample 6_MOLM14_6H - Genes after coverage filtering: 2712
Sample 9_MOLM14_6H - Genes after coverage filtering: 2754
Sample MOLM14-SRRs - TOTAL genes after ALL coverage filtering: 2583
Processing experiment: Meta-MOLM14
Sample meta_MOLM14_V - Genes after coverage filtering: 2873
Sample meta_MOLM14_1H - Genes after coverage filtering: 2902


/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_threshold['norm_complexity'] = df_threshold[7]/df_threshold[8]
/tmp/ipykernel_393780/3605895782.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

Sample meta_MOLM14_6H - Genes after coverage filtering: 2932
Sample Meta-MOLM14 - TOTAL genes after ALL coverage filtering: 2815


In [50]:
450740 - 451366

-626

# Gene Length Filtering

In [51]:
# 2. Length filter before running LIET

# Outdir will be the filtered-genes dir
outdir = '/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/filtered-genes/'

# MANE file is annotation
annotation_file = '/scratch/Shares/dowell/geba9152/MANE/transcript.MANE.refseq.bed'

def length_filter_genes(metadata_df, outdir, annotation_file):
    """
     Gene length filtering, LIET performs poorly on really short genes
    
    """
    
    for _, experiment in metadata_df.iterrows():
        experiment_name = experiment['Experiment']
        samples = experiment['Samples'].split(',')
        coverage_threshold = experiment['Sense_Coverage_Cutoff']
        gene_len_cutoff = experiment['Gene_Length_Cutoff']
        
        # Load filtered genes for this experiment
        filtered_file_path = os.path.join(outdir, f"{experiment_name}/isolated_{coverage_threshold}_coverage_filtered_genes.txt")
        
        if not os.path.exists(filtered_file_path):
            print(f"Filtered file not found for {experiment_name}, skipping.")
            continue
        
        # Read the filtered genes
        filtered_genes = pd.read_csv(filtered_file_path, sep="\t", header=None, names=["Gene"])
        starting_len = len(filtered_genes)
        
        print(f"Starting length of gene list: {starting_len}")
        
        # Read in ann
        ann = pd.read_csv(annotation_file, sep = "\t", header = None)
        ann.columns = ['chr', 'start','stop','Gene','.','strand']
        ann['Length'] = abs(ann['start'] - ann['stop'])
        
        # Merge
        merge = ann.merge(filtered_genes, on = 'Gene' )
        merge_len = len(merge)
        
#         print(f"Merged length of gene list: {merge_len}")
        
        # Filter based on thresh
        df_filtered = merge[merge["Length"] >= gene_len_cutoff]
                
        df_filtered_len = len(df_filtered)
    
        print(f"Sample {experiment_name} - TOTAL genes after coverage/length filtering: {df_filtered_len}")

        df_filtered = df_filtered['Gene']
        
        # Save
        output_file = os.path.join(outdir, f"{experiment_name}/isolated_{coverage_threshold}_coverage_length_filtered_genes.txt")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        df_filtered.to_csv(output_file, index=False, sep="\t", header=None)

length_filter_genes(metadata_df, outdir, annotation_file)

Starting length of gene list: 2582
Sample Meta-Eric-IFN-Beta - TOTAL genes after coverage/length filtering: 2299
Starting length of gene list: 2387
Sample OV90-SRRs - TOTAL genes after coverage/length filtering: 2123
Starting length of gene list: 2396
Sample Daniel-IFN - TOTAL genes after coverage/length filtering: 2165
Starting length of gene list: 3438
Sample Meta-BEAS2B-Woodsmoke - TOTAL genes after coverage/length filtering: 3037
Starting length of gene list: 1905
Sample OV90-CDK7i - TOTAL genes after coverage/length filtering: 1692
Starting length of gene list: 3631
Sample U2OS-Specific - TOTAL genes after coverage/length filtering: 3140
Starting length of gene list: 3384
Sample Meta-Eric-HS - TOTAL genes after coverage/length filtering: 2968
Starting length of gene list: 2978
Sample Nina-Arsenic - TOTAL genes after coverage/length filtering: 2606
Starting length of gene list: 3214
Sample Meta-Jaegar2020-CDK9i - TOTAL genes after coverage/length filtering: 2867
Starting length of 

# Generate LIET Annotation Files 

In [58]:
# Filter metadata based on pad/annotations we are making
metadata_df = pd.read_csv("/scratch/Users/geba9152/LIET-3end-analysis/LIET-Gene-Filtering/metadata/metadata.txt", sep = "\t")

samps_lengthen_3end = ["Eric-HS","Meta-Eric-HS",
                       "OV90-CDK7i", "Meta-OV90-CDK7i",
                       "OV90-HS","Meta-OV90-HS",
                       "Nina-Arsenic","Meta-Nina-Arsenic",
                       "Birkenheuer2018herpes","Meta-Birkenheuer2018herpes",
                       "OV90-CDK7i-HS", "Meta-OV90-CDK7i-HS"]


# # Filter for ONLY 3 lengtheners
# 30 kb pads
# metadata_df = metadata_df[metadata_df['Experiment'].isin(samps_lengthen_3end)]
# metadata_df

# Filter NON 3 lengtheners
# 20 kb pads
metadata_df = metadata_df[~metadata_df['Experiment'].isin(samps_lengthen_3end)]
metadata_df


,Experiment,Samples,Sample_Count,Colors,Sense_Coverage_Cutoff,Gene_Length_Cutoff,Annotation,Residual_Cutoff,wB_Cutoff,wB_a_Cutoff,Coverage_Antisense_Cutoff,Names,Normalized_Complexity_Cutoff
0,Meta-Eric-IFN-Beta,"Meta-Eric-IFN-Beta_meta_PRO-BSA-Eric,Meta-Eric...",2,"#dededd,#defaab",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"meta_PRO-BSA-Eric,meta_PRO-IFNB-Eric",0.001
1,OV90-SRRs,"OV90-SRRs_DMSO_0_R1,OV90-SRRs_DMSO_0_R2",2,"#ffc49c,#ffc49c",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"DMSO_0_R1,DMSO_0_R2",0.001
2,Daniel-IFN,"Daniel-IFN_PRO-BSA-Human-1,Daniel-IFN_PRO-BSA-...",4,"#dededd,#dededd,#9ad66b,#9ad66b",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"PRO-BSA-Human-1,PRO-BSA-Human-2,PRO-IFN-Human-...",0.001
3,Meta-BEAS2B-Woodsmoke,Meta-BEAS2B-Woodsmoke_meta_BEAS2B-Woodsmoke-Ve...,3,"#dededd,#8dcdeb,#59a4c8",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"meta_BEAS2B-Woodsmoke-Veh,meta_BEAS2B-Woodsmok...",0.001
5,U2OS-Specific,U2OS-Specific_meta_Nina-Control,1,#f3b26c,0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,meta_Nina-Control,0.001
8,Meta-Jaegar2020-CDK9i,"Meta-Jaegar2020-CDK9i_meta_Jaegar2020-Control,...",2,"#dededd,#b475f4",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"meta_Jaegar2020-Control,meta_Jaegar2020-CDK9i",0.001
9,Eric-IFN-Beta,"Eric-IFN-Beta_PRO-BSA-Eric-1,Eric-IFN-Beta_PRO...",4,"#dededd,#dededd,#defaab,#defaab",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"PRO-BSA-Eric-1,PRO-BSA-Eric-2,PRO-IFNB-Eric-1,...",0.001
10,Jaegar2020-CDK9i,"Jaegar2020-CDK9i_SRZ10354600,Jaegar2020-CDK9i_...",4,"#dededd,#dededd,#b475f4,#b475f4",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"SRZ10354600,SRZ10354602,SRZ10354616,SRZ10354618",0.001
11,HCT-SRRs,"HCT-SRRs_SRZ6290515,HCT-SRRs_SRZ6290523,HCT-SR...",10,"#eabf81,#eabf81,#eabf81,#eabf81,#eabf81,#eabf8...",0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,"SRZ6290515,SRZ6290523,SRR9304731,SRR8867637,SR...",0.001
14,BEAS2B-Specific,BEAS2B-Specific_meta_BEAS2B,1,#ffd064,0.001,7000,/scratch/Users/geba9152/LIET-3end-analysis/gen...,13,0.5,0.7,0.005,meta_BEAS2B,0.001


In [59]:
# 3. Generate 3p UTR reference LIET annotation 

# outdir will be annotation folder
outdir = "/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/annotations/"

def make_3_p_liet_ann(metadata_df, outdir):
    
    for _, experiment in metadata_df.iterrows():
        experiment_name = experiment['Experiment']
        samples = experiment['Samples'].split(',')
        coverage_threshold = experiment['Sense_Coverage_Cutoff']
        
        # read in transcript annotation file (has 3p UTR coordinates)
        transcript_annotation_file = '/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/annotations/master_MANE_Andrew_Jacob_transcripts.bed'
        transcript_ann = pd.read_csv(transcript_annotation_file, sep = "\t", header = None, names=['chr','start','stop','gene','.','strand'])

        # load in filtered genes
        filtereddir = '/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/filtered-genes/'
        filtered_file_path = os.path.join(filtereddir, f"{experiment_name}/isolated_{coverage_threshold}_coverage_length_filtered_genes.txt")
        filtered_genes = pd.read_csv(filtered_file_path, sep="\t", header=None, names=["gene"])

        # merge w annotatation
        filtered_genes_ann = filtered_genes.merge(transcript_ann, on = "gene")
        filtered_genes_ann['length'] = abs(filtered_genes_ann['stop'] - filtered_genes_ann['start'])
        filtered_genes_ann['just-gene'] = filtered_genes['gene'].str.split("|").str[0]
                
        # Check for duplicates in the 'just-gene' column
        # makes sure one gene per isoform
        duplicates = filtered_genes_ann[filtered_genes_ann['just-gene'].duplicated(keep=False)]
        
        # If duplicates exist, you can handle them (e.g., remove or inspect)
        if not duplicates.empty:
            print("Duplicate genes found:")
            print(duplicates)
        
        # redorder & sort
        filtered_genes_liet_ann = filtered_genes_ann[['chr','start','stop','gene','length','strand']]
        filtered_genes_liet_ann = filtered_genes_liet_ann.sort_values(by=['chr', 'start'])
        
        print(f"Sample {experiment_name} - Genes in liet.ann: {len(filtered_genes_liet_ann)}")
        
        # save
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-3p-UTR.liet.ann")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        filtered_genes_liet_ann.to_csv(output_file, index=False, sep="\t", header=None)
        
        # redorder & sort for bed format
        filtered_genes_bed = filtered_genes_ann[['chr','start','stop','gene','.','strand']]
        filtered_genes_bed = filtered_genes_bed.sort_values(by=['chr', 'start'])
        
        # save
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-3p-UTR.bed")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        filtered_genes_bed.to_csv(output_file, index=False, sep="\t", header=None)
        
make_3_p_liet_ann(metadata_df, outdir)     


Sample Meta-Eric-IFN-Beta - Genes in liet.ann: 2299
Sample OV90-SRRs - Genes in liet.ann: 2123
Sample Daniel-IFN - Genes in liet.ann: 2165
Sample Meta-BEAS2B-Woodsmoke - Genes in liet.ann: 3037
Sample U2OS-Specific - Genes in liet.ann: 3140
Sample Meta-Jaegar2020-CDK9i - Genes in liet.ann: 2867
Sample Eric-IFN-Beta - Genes in liet.ann: 665
Sample Jaegar2020-CDK9i - Genes in liet.ann: 2467
Sample HCT-SRRs - Genes in liet.ann: 2458
Sample BEAS2B-Specific - Genes in liet.ann: 3020
Sample BEAS2B-Woodsmoke - Genes in liet.ann: 2708
Sample LCL-Specfic - Genes in liet.ann: 3968
Sample BEAS2B-SRRs - Genes in liet.ann: 2501
Sample HCT-Specific - Genes in liet.ann: 3837
Sample LCL-SRRs - Genes in liet.ann: 2130
Sample KBM7-SRRs - Genes in liet.ann: 2661
Sample K562-Specific - Genes in liet.ann: 3973
Sample U2OS-SRRs - Genes in liet.ann: 2690
Sample Meta-Daniel-IFN - Genes in liet.ann: 2429
Sample KBM7-Specific - Genes in liet.ann: 3853
Sample K562-SRRs - Genes in liet.ann: 1827
Sample OV90-Speci

In [60]:
# 4. Generate 5p UTR reference LIET annotation & pad file 

# outdir will be annotation folder
outdir = "/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/annotations/"

def make_5_p_liet_ann_pad(metadata_df, outdir, threeprimepad):
    
    for _, experiment in metadata_df.iterrows():
        experiment_name = experiment['Experiment']
        samples = experiment['Samples'].split(',')
        coverage_threshold = experiment['Sense_Coverage_Cutoff']
        
        # read in MANE exon file
        ex_annotation_file = '/scratch/Users/geba9152/LIET-3end-analysis/gene_curation_MANE/annotations/master_MANE_Andrew_Jacob_exons.bed'
        ann_ex = pd.read_csv(ex_annotation_file, sep = "\t", header = None, names=['chr','start','stop','gene','.','strand'])

        # load in filtered genes
        threep_UTR = f'{outdir}{experiment_name}/{experiment_name}-3p-UTR.liet.ann'
        threep_UTR = pd.read_csv(threep_UTR, sep="\t", header=None, names=['chr','start','stop','gene','length','strand'])
        print(len(threep_UTR))
        
        filtered_genes = threep_UTR['gene'].to_list()

        # make a last_exon_coords df
        last_exon_coordinates = pd.DataFrame(columns=ann_ex.columns)
    
        # select only isoforms we are interested in
        selected_ann_ex = ann_ex.loc[ann_ex["gene"].isin(filtered_genes)]
        print(len(filtered_genes))
                
        # loop through df (group) and gene (roi)
        for roi, group in selected_ann_ex.groupby('gene'):
            if group['strand'].iloc[0] == "+":
                sorted_stops = group.sort_values(by='stop', ascending=False)
                # Note "last_exon == UTR"
                last_exon = sorted_stops.iloc[0]

            elif group['strand'].iloc[0] == "-":
                sorted_starts = group.sort_values(by='start', ascending=True)
                last_exon = sorted_starts.iloc[0]

            # Concatenate the selected exon to the DataFrame
            last_exon_coordinates = pd.concat([last_exon_coordinates, pd.DataFrame([last_exon])], ignore_index=True)
        
        # Organize after all the concatenations
        last_exon_coordinates = last_exon_coordinates.sort_values(by=["chr", "start"])
        last_exon_coordinates.columns = ["chr","start-ex","stop-ex","gene",".","strand"]
        last_exon_coordinates = last_exon_coordinates[["start-ex","stop-ex","gene"]]
        
        # Merge with LIET annotation to get correct structure
        fivep_UTR = last_exon_coordinates.merge(threep_UTR, on="gene")
        print(len(fivep_UTR))
                
        # Make copy for pad 
        fivep_UTR_pad = fivep_UTR.copy()
        
        fivep_UTR['start'] = fivep_UTR.apply(lambda row: row["stop-ex"] if row["strand"] == "-" else row['start'], axis=1)
        fivep_UTR['stop'] = fivep_UTR.apply(lambda row: row["start-ex"] if row["strand"] == "+" else row['stop'], axis=1)
        
        # Check that len of 5p and 3p files are same
        if len(threep_UTR) != len(fivep_UTR):
            raise ValueError(
            f"Length mismatch: threep_UTR has {len(threep_UTR)} rows, "
            f"but merged result has {len(fivep_UTR)} rows."
        )
        
        fivep_UTR_ann = fivep_UTR[['chr','start','stop','gene','length','strand']]
        fivep_UTR_ann = fivep_UTR_ann.sort_values(by=['chr', 'start'])
        
        # save
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-5p-UTR.liet.ann")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        fivep_UTR_ann.to_csv(output_file, index=False, sep="\t", header=None)
        
        # redorder & sort for bed format
        fivep_UTR_ann['.'] = '.'
        filtered_genes_bed = fivep_UTR_ann[['chr','start','stop','gene','.','strand']]
        filtered_genes_bed = filtered_genes_bed.sort_values(by=['chr', 'start'])
        
        # save
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-5p-UTR.bed")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        filtered_genes_bed.to_csv(output_file, index=False, sep="\t", header=None)
        
        # pad files 
        fivep_UTR_pad['UTR-len'] = fivep_UTR_pad.apply(
            lambda row: abs(row["stop"] - row["start-ex"]) if row["strand"] == "+" else abs(row["stop-ex"] - row["start"]),
            axis=1
        )
                
        # Calculate pads
        fivep_UTR_pad['3prime-end-pad'] = fivep_UTR_pad['UTR-len'] + threeprimepad
        fivep_UTR_pad['5prime-end-pad'] = 3000
        fivep_UTR_pad['pad'] = fivep_UTR_pad['5prime-end-pad'].astype(str) + ',' + fivep_UTR_pad['3prime-end-pad'].astype(str)
                
        # Sort and save 
        # Reorder fivep_UTR_pad to match the gene order in fivep_UTR_ann
        fivep_UTR_pad = fivep_UTR_pad.set_index('gene')
        print(f"{len(fivep_UTR_pad)}")
        fivep_UTR_pad = fivep_UTR_pad.loc[fivep_UTR_ann['gene']].reset_index()
        fivep_UTR_pad = fivep_UTR_pad.drop_duplicates()
        fivep_UTR_pad = fivep_UTR_pad[['gene','pad']]
        print(f"{len(fivep_UTR_pad)}")
                
        # Check that len of pad and 5p files are same
        if len(fivep_UTR_pad) != len(fivep_UTR):
            raise ValueError(
            f"Length mismatch: fivep_UTR_pad has {len(fivep_UTR_pad)} rows, "
            f"but merged result has {len(fivep_UTR)} rows."
        )
        
        output_file = os.path.join(outdir, f"{experiment_name}/{experiment_name}-5p-UTR.pad")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        fivep_UTR_pad.to_csv(output_file, index=False, sep="\t", header=None)
        
# threeprimepad = 30000
threeprimepad = 20000

make_5_p_liet_ann_pad(metadata_df, outdir, threeprimepad)

2299
2299
2299
2299
2299
2123
2123
2123
2123
2123
2165
2165
2165
2165
2165
3037
3037
3037
3037
3037
3140
3140
3140
3140
3140
2867
2867
2867
2867
2867
665
665
665
665
665
2467
2467
2467
2467
2467
2458
2458
2458
2458
2458
3020
3020
3020
3020
3020
2708
2708
2708
2708
2708
3968
3968
3968
3968
3968
2501
2501
2501
2501
2501
3837
3837
3837
3837
3837
2130
2130
2130
2130
2130
2661
2661
2661
2661
2661
3973
3973
3973
3973
3973
2690
2690
2690
2690
2690
2429
2429
2429
2429
2429
3853
3853
3853
3853
3853
1827
1827
1827
1827
1827
2545
2545
2545
2545
2545
3429
3429
3429
3429
3429
2544
2544
2544
2544
2544
2529
2529
2529
2529
2529
2163
2163
2163
2163
2163
2052
2052
2052
2052
2052
2426
2426
2426
2426
2426
2226
2226
2226
2226
2226
2568
2568
2568
2568
2568
2563
2563
2563
2563
2563
2891
2891
2891
2891
2891
2369
2369
2369
2369
2369
2753
2753
2753
2753
2753
2529
2529
2529
2529
2529
2919
2919
2919
2919
2919
3258
3258
3258
3258
3258
2363
2363
2363
2363
2363
21
21
21
21
21
1531
1531
1531
1531
1531
3961
3961
3961
